In [5]:
import sys
from pathlib import Path

project_root = Path().cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root = Path().cwd().parent

from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

r = simulate(["double_spell", "spark_bolt", "spark_bolt"], wand, TargetInfo(distance_px=300))
print("═══ 核心 ═══")
print(f"DPS: {r.dps:.1f}")
print(f"总伤害: {r.total_damage:.1f}")
print(f"总发射: {r.total_projectiles}")
print(f"模拟时长: {r.total_time_simulated:.1f}s")

print("═══ 续航 ═══")
print(f"法力耗尽时间: {r.mana_exhaustion_time:.1f}s  {'✓ 永动' if r.mana_exhaustion_time >= r.total_time_simulated else '✗ 耗尽'}")
print(f"法力消耗: {r.total_mana_spent:.0f}")
print(f"每秒耗蓝: {r.mana_usage_per_second:.1f}")

print("═══ 质量 ═══")
print(f"单发伤害: {r.avg_damage_per_hit:.2f}")
print(f"命中率: {r.hit_rate:.1%}")
print(f"暴击占比: {r.crit_ratio:.1%}")

print("═══ 调试 ═══")
print(f"总轮数: {r.total_rounds}")
print(f"每轮耗时: {r.avg_round_time:.2f}s")
print(f"发射时间占比: {r.firing_uptime:.1%}")
print(f"每秒发射: {r.avg_projectiles_per_second:.1f}")

═══ 核心 ═══
DPS: 4.3
总伤害: 44.6
总发射: 38
模拟时长: 10.5s
═══ 续航 ═══
法力耗尽时间: 10.5s  ✓ 永动
法力消耗: 190
每秒耗蓝: 18.2
═══ 质量 ═══
单发伤害: 1.17
命中率: 32.6%
暴击占比: 16.7%
═══ 调试 ═══
总轮数: 19
每轮耗时: 0.55s
发射时间占比: 40.0%
每秒发射: 3.6


## 嵌套多重 + 修正 + Chainsaw 验证

In [6]:
import sys
from pathlib import Path
project_root = Path().cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

# 修正 + 多重：伤害增强应在组内对所有投射物生效
r1 = simulate(["double_spell", "spark_bolt", "spark_bolt"], wand)
r2 = simulate(["double_spell", "damage_plus", "spark_bolt", "spark_bolt"], wand)

print("无修正  vs  有伤害增强")
print(f"DPS:     {r1.dps:.1f}   vs   {r2.dps:.1f} {'✓ 增加' if r2.dps > r1.dps else '✗ 异常'}")
print(f"单发伤害: {r1.avg_damage_per_hit:.2f}  vs   {r2.avg_damage_per_hit:.2f} {'✓ 增加' if r2.avg_damage_per_hit > r1.avg_damage_per_hit else '✗ 异常'}")
print(f"每秒发射: {r1.avg_projectiles_per_second:.1f}  vs   {r2.avg_projectiles_per_second:.1f}  (应相等)")

无修正  vs  有伤害增强
DPS:     4.3   vs   18.5 ✓ 增加
单发伤害: 1.17  vs   5.08 ✓ 增加
每秒发射: 3.6  vs   3.6  (应相等)


In [7]:
# Chainsaw 机枪：充能时间 -0.17s 应显著降低每轮耗时
r1 = simulate(["double_spell", "spark_bolt", "spark_bolt"], wand)
r2 = simulate(["double_spell", "chainsaw", "spark_bolt"], wand)

print("无 chainsaw   vs   有 chainsaw")
print(f"DPS:       {r1.dps:.1f}       vs   {r2.dps:.1f} {'✓ 增加' if r2.dps > r1.dps else '✗ 异常'}")
print(f"每轮耗时:   {r1.avg_round_time:.2f}s     vs   {r2.avg_round_time:.2f}s {'✓ 缩短' if r2.avg_round_time < r1.avg_round_time else '✗ 异常'}")
print(f"每秒发射:   {r1.avg_projectiles_per_second:.1f}       vs   {r2.avg_projectiles_per_second:.1f} {'✓ 增加' if r2.avg_projectiles_per_second > r1.avg_projectiles_per_second else '✗ 异常'}")
print(f"暴击占比:   {r1.crit_ratio:.1%}       vs   {r2.crit_ratio:.1%}")
print(f"总轮数:     {r1.total_rounds}         vs   {r2.total_rounds}")

无 chainsaw   vs   有 chainsaw
DPS:       4.3       vs   8.9 ✓ 增加
每轮耗时:   0.55s     vs   0.38s ✓ 缩短
每秒发射:   3.6       vs   5.3 ✓ 增加
暴击占比:   16.7%       vs   5.8%
总轮数:     19         vs   27


In [8]:
# 嵌套多重 + 修正：damage_plus 应对组内全部 4 发生效
r1 = simulate(["double_spell", "triple_spell", "spark_bolt", "spark_bolt", "spark_bolt", "spark_bolt"], wand)
r2 = simulate(["double_spell", "triple_spell", "damage_plus", "spark_bolt", "spark_bolt", "spark_bolt", "spark_bolt"], wand)

print("嵌套无修正  vs  嵌套有伤害增强")
print(f"DPS:     {r1.dps:.1f}   vs   {r2.dps:.1f} {'✓ 增加' if r2.dps > r1.dps else '✗ 异常'}")
print(f"单发伤害: {r1.avg_damage_per_hit:.2f}  vs   {r2.avg_damage_per_hit:.2f} {'✓ 增加' if r2.avg_damage_per_hit > r1.avg_damage_per_hit else '✗ 异常'}")
print(f"总轮数:   {r1.total_rounds}         vs   {r2.total_rounds}         (应相等)")
print(f"每秒发射: {r1.avg_projectiles_per_second:.1f}  vs   {r2.avg_projectiles_per_second:.1f}  (应相等)")

嵌套无修正  vs  嵌套有伤害增强
DPS:     8.4   vs   36.5 ✓ 增加
单发伤害: 1.17  vs   5.08 ✓ 增加
总轮数:   19         vs   19         (应相等)
每秒发射: 7.2  vs   7.2  (应相等)


In [9]:
import sys
from pathlib import Path

project_root = Path().cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root = Path().cwd().parent

from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

r = simulate(["double_spell", "triple_spell", "spark_bolt", "spark_bolt", "spark_bolt", "spark_bolt"], wand, TargetInfo(distance_px=300))
print("═══ 核心 ═══")
print(f"DPS: {r.dps:.1f}")
print(f"总伤害: {r.total_damage:.1f}")
print(f"总发射: {r.total_projectiles}")
print(f"模拟时长: {r.total_time_simulated:.1f}s")

print("═══ 续航 ═══")
print(f"法力耗尽时间: {r.mana_exhaustion_time:.1f}s  {'✓ 永动' if r.mana_exhaustion_time >= r.total_time_simulated else '✗ 耗尽'}")
print(f"法力消耗: {r.total_mana_spent:.0f}")
print(f"每秒耗蓝: {r.mana_usage_per_second:.1f}")

print("═══ 质量 ═══")
print(f"单发伤害: {r.avg_damage_per_hit:.2f}")
print(f"命中率: {r.hit_rate:.1%}")
print(f"暴击占比: {r.crit_ratio:.1%}")

print("═══ 调试 ═══")
print(f"总轮数: {r.total_rounds}")
print(f"每轮耗时: {r.avg_round_time:.2f}s")
print(f"发射时间占比: {r.firing_uptime:.1%}")
print(f"每秒发射: {r.avg_projectiles_per_second:.1f}")

═══ 核心 ═══
DPS: 8.4
总伤害: 88.0
总发射: 75
模拟时长: 10.5s
═══ 续航 ═══
法力耗尽时间: 9.9s  ✗ 耗尽
法力消耗: 375
每秒耗蓝: 35.9
═══ 质量 ═══
单发伤害: 1.17
命中率: 32.6%
暴击占比: 16.7%
═══ 调试 ═══
总轮数: 19
每轮耗时: 0.55s
发射时间占比: 40.0%
每秒发射: 7.2
